In [7]:
ls data/UD_Turkish-BOUN/

tr_boun-ud-dev.conllu  tr_boun-ud-test.conllu  tr_boun-ud-train.conllu


In [14]:
from collections import defaultdict
from typing import List, Dict, Any, Optional

def parse_conllu_file(filepath: str) -> List[Dict[str, Any]]:
    """
    Parse a CoNLL-U file and return a list of sentences.
    
    Each sentence is a dict with:
        - 'sent_id': sentence ID
        - 'text': original sentence text
        - 'tokens': list of token dicts with CoNLL-U fields
    
    Token fields: id, form, lemma, upos, xpos, feats, head, deprel, deps, misc
    """
    sentences = []
    current_sentence = {'tokens': []}
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if not line:
                # Empty line marks end of sentence
                if current_sentence['tokens']:
                    sentences.append(current_sentence)
                current_sentence = {'tokens': []}
                continue
            
            if line.startswith('#'):
                # Metadata line
                if line.startswith('# sent_id'):
                    current_sentence['sent_id'] = line.split('=', 1)[1].strip()
                elif line.startswith('# text'):
                    current_sentence['text'] = line.split('=', 1)[1].strip()
                continue
            
            # Token line - 10 tab-separated fields
            fields = line.split('\t')
            if len(fields) != 10:
                continue
                
            token_id, form, lemma, upos, xpos, feats, head, deprel, deps, misc = fields
            
            # Skip multi-word token lines (e.g., "6-7")
            if '-' in token_id:
                continue
            
            # Parse features into dict
            feats_dict = {}
            if feats != '_':
                for feat in feats.split('|'):
                    if '=' in feat:
                        key, value = feat.split('=', 1)
                        feats_dict[key] = value
            
            token = {
                'id': token_id,
                'form': form,
                'lemma': lemma,
                'upos': upos,  # Universal POS tag
                'xpos': xpos,  # Language-specific POS tag
                'feats': feats_dict,  # Morphological features as dict
                'feats_str': feats,   # Original feature string
                'head': head,
                'deprel': deprel,
                'deps': deps,
                'misc': misc
            }
            current_sentence['tokens'].append(token)
        
        # Don't forget the last sentence
        if current_sentence['tokens']:
            sentences.append(current_sentence)
    
    return sentences


def load_ud_dataset(base_path: str = "data/UD_Turkish-BOUN") -> Dict[str, List[Dict]]:
    """
    Load all splits of the UD Turkish BOUN dataset.
    
    Returns:
        dict with 'train', 'dev', 'test' keys, each containing list of sentences
    """
    import os
    
    from datasets import Dataset, DatasetDict

    splits = {
        'train': 'tr_boun-ud-train.conllu',
        'dev': 'tr_boun-ud-dev.conllu',
        'test': 'tr_boun-ud-test.conllu'
    }

    datasets_dict = {}
    for split_name, filename in splits.items():
        filepath = os.path.join(base_path, filename)
        if os.path.exists(filepath):
            parsed_sentences = parse_conllu_file(filepath)
            datasets_dict[split_name] = Dataset.from_list(parsed_sentences)
            print(f"Loaded {split_name}: {len(parsed_sentences)} sentences")

    dataset = DatasetDict(datasets_dict)

    return dataset

In [15]:
# Load the UD Turkish BOUN dataset
ud_dataset = load_ud_dataset("data/UD_Turkish-BOUN")

Loaded train: 7803 sentences
Loaded dev: 979 sentences
Loaded test: 979 sentences


In [16]:
ud_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 7803
    })
    dev: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 979
    })
    test: Dataset({
        features: ['tokens', 'sent_id', 'text'],
        num_rows: 979
    })
})

In [20]:
ud_dataset['train'][100]

{'tokens': [{'deprel': 'root',
   'deps': '_',
   'feats': {'Abbr': None,
    'Aspect': None,
    'Case': 'Loc',
    'Echo': None,
    'Evident': None,
    'Mood': None,
    'NumType': None,
    'Number': 'Sing',
    'Number[psor]': None,
    'Person': '3',
    'Person[psor]': None,
    'Polarity': None,
    'PronType': None,
    'Reflex': None,
    'Tense': None,
    'Typo': None,
    'VerbForm': None,
    'Voice': None},
   'feats_str': 'Case=Loc|Number=Sing|Person=3',
   'form': 'Devede',
   'head': '0',
   'id': '1',
   'lemma': 'deve',
   'misc': '_',
   'upos': 'NOUN',
   'xpos': '_'},
  {'deprel': 'compound',
   'deps': '_',
   'feats': {'Abbr': None,
    'Aspect': None,
    'Case': 'Nom',
    'Echo': None,
    'Evident': None,
    'Mood': None,
    'NumType': None,
    'Number': 'Sing',
    'Number[psor]': None,
    'Person': '3',
    'Person[psor]': None,
    'Polarity': None,
    'PronType': None,
    'Reflex': None,
    'Tense': None,
    'Typo': None,
    'VerbForm': None,


In [4]:
import sys
import typing

typing.io = typing
sys.modules.setdefault("typing.io", typing) 

from zemberek import TurkishMorphology
morphology = TurkishMorphology.create_with_defaults()

# SINGLE WORD MORPHOLOGICAL ANALYSIS
results = morphology.analyze("kalemin")
for result in results:
    print(result)
print("\n")

# SENTENCE ANALYSIS AND DISAMBIGUATION
sentence = "Yarın kar yağacak."
analysis = morphology.analyze_sentence(sentence)
after = morphology.disambiguate(sentence, analysis)

print("\nBefore disambiguation")
for e in analysis:
    print(f"Word = {e.inp}")
    for s in e:
        print(s.format_string())

print("\nAfter disambiguation")
for s in after.best_analysis():
    print(s.format_string())


text = "İnsanoğlu aslında ne para ne sevgi ne kariyer ne şöhret ne de çevre ile sonsuza dek mutlu olabilecek bir " \
       "yapıya sahiptir. Dış kaynaklardan gelebilecek bu mutluluklar sadece belirli bir zaman için insanı mutlu " \
       "kılıyor. Kişi bu kaynakları elde ettiği zaman belirli bir dönem için kendini iyi hissediyor, ancak alışma " \
       "dönemine girdiği andan itibaren bu iyilik hali hızla tükeniyor. Mutlu olma sanatının özü bu değildir. Gerçek " \
       "mutluluk, kişinin her türlü olaya ve duruma karşı kendini pozitif tutarak mutlu hissedebilmesi halidir. Bu " \
       "davranış şeklini edinen insan, zor günlerde güçlü, mutlu günlerde zevk alan biri olur ve mutluluğu kalıcı " \
       "kılar. "
analysis = morphology.analyze_sentence(text)
after = morphology.disambiguate(text, analysis)

print("\nBefore disambiguation")
for e in analysis:
    print(f"Word = {e.inp}")
    for s in e:
        print(s.format_string())

print("\nAfter disambiguation")
for s in after.best_analysis():
    print(s.format_string())

2026-02-21 12:06:20,501 - zemberek.morphology.turkish_morphology - INFO
Msg: TurkishMorphology instance initialized in 2.9699904918670654

[Kale:Noun, Prop] kale:Noun+A3sg+m:P1sg+in:Gen
[kale:Noun] kale:Noun+A3sg+m:P1sg+in:Gen
[kalem:Noun] kalem:Noun+A3sg+in:Gen
[kalem:Noun] kalem:Noun+A3sg+in:P2sg



Before disambiguation
Word = Yarın
[yarın:Adv] yarın:Adv
[yarmak:Verb] yar:Verb+Imp+ın:A2pl
[yar:Noun] yar:Noun+A3sg+ın:Gen
[yar:Noun] yar:Noun+A3sg+ın:P2sg
[yarı:Noun] yarı:Noun+A3sg+n:P2sg
[yarın:Noun, Time] yarın:Noun+A3sg
[yarı:Adj] yarı:Adj|Zero→Noun+A3sg+n:P2sg
Word = kar
[karmak:Verb] kar:Verb+Imp+A2sg
[kar:Noun] kar:Noun+A3sg
[kâr:Noun] kar:Noun+A3sg
Word = yağacak
[yağmak:Verb] yağ:Verb+acak:Fut+A3sg
[yağmak:Verb] yağ:Verb|acak:FutPart→Adj
Word = .
[.:Punc] .:Punc

After disambiguation
[yarın:Noun, Time] yarın:Noun+A3sg
[kar:Noun] kar:Noun+A3sg
[yağmak:Verb] yağ:Verb+acak:Fut+A3sg
[.:Punc] .:Punc

Before disambiguation
Word = İnsanoğlu
Word = aslında
[aslında:Adv] aslında:Adv
[as

In [2]:
!python --version

Python 3.14.2


In [35]:
# Required for zeyrek (Turkish morphological analyzer) - run once
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/hpc/b279bb/b279bb15/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import zeyrek
zeyrek_analyzer = zeyrek.MorphAnalyzer()
for parse in zeyrek_analyzer.analyze('benim')[0]:
    print(parse)

2026-02-15 17:48:43,327 - zeyrek.rulebasedanalyzer - WARNING
Msg: APPENDING RESULT: <(ben_Pron_Pers)(-)(ben:pronPers_S + pA1sg_S + pPnon_S + im:pGen_ST)>

2026-02-15 17:48:43,327 - zeyrek.rulebasedanalyzer - WARNING
Msg: APPENDING RESULT: <(ben_Noun)(-)(ben:noun_S + a3sg_S + im:p1sg_S + nom_ST)>

2026-02-15 17:48:43,328 - zeyrek.rulebasedanalyzer - WARNING
Msg: APPENDING RESULT: <(ben_Pron_Pers)(-)(ben:pronPers_S + pA1sg_S + pPnon_S + pNom_ST + pronZeroDeriv_S + pvVerbRoot_S + pvPresent_S + im:pvA1sg_ST)>

2026-02-15 17:48:43,328 - zeyrek.rulebasedanalyzer - WARNING
Msg: APPENDING RESULT: <(ben_Noun)(-)(ben:noun_S + a3sg_S + pnon_S + nom_ST + nounZeroDeriv_S + nVerb_S + nPresent_S + im:nA1sg_ST)>

Parse(word='benim', lemma='ben', pos='Pron', morphemes=['Pron', 'A1sg', 'Gen'], formatted='[ben:Pron,Pers] ben:Pron+A1sg+im:Gen')
Parse(word='benim', lemma='ben', pos='Noun', morphemes=['Noun', 'A3sg', 'P1sg'], formatted='[ben:Noun] ben:Noun+A3sg+im:P1sg')
Parse(word='benim', lemma='ben', pos

In [1]:
from vnlp import StemmerAnalyzer
stemmer = StemmerAnalyzer()
stemmer.predict("Üniversite sınavlarına canla başla çalışıyorlardı.")

# ['üniversite+Noun+A3sg+Pnon+Nom',
# 'sınav+Noun+A3pl+P3sg+Dat',
# 'can+Noun+A3sg+Pnon+Ins',
# 'baş+Noun+A3sg+Pnon+Ins',
# 'çalış+Verb+Pos+Prog1+A3pl+Past',
# '.+Punc']

2026-02-15 22:20:06.306965: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-15 22:20:06.342535: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-15 22:20:06.342578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-15 22:20:06.343970: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-15 22:20:06.350434: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-15 22:20:06.351646: I tensorflow/core/platform/cpu_feature_guard.cc:1

['üniversite+Noun+A3sg+Pnon+Nom',
 'sınav+Noun+A3pl+P3sg+Dat',
 'can+Noun+A3sg+Pnon+Ins',
 'baş+Noun+A3sg+Pnon+Ins',
 'çalış+Verb+Pos+Prog1+A3pl+Past',
 '.+Punc']

In [ ]:
import spacy
from spacy.lang.de.examples import sentences 

nlp = spacy.load("de_core_news_lg")

In [ ]:
text = "Ich gebe dem Mann das Buch."
doc = nlp(text)

for token in doc:
    print(token.text)
    print("  POS:", token.pos_)
    print("  TAG:", token.tag_)
    print("  MORPH:", token.morph)
    print()

Ich
  POS: PRON
  TAG: PPER
  MORPH: Case=Nom|Number=Sing|Person=1|PronType=Prs

gebe
  POS: VERB
  TAG: VVFIN
  MORPH: Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin

dem
  POS: DET
  TAG: ART
  MORPH: Case=Dat|Definite=Def|Gender=Masc|Number=Sing|PronType=Art

Mann
  POS: NOUN
  TAG: NN
  MORPH: Case=Dat|Gender=Masc|Number=Sing

das
  POS: DET
  TAG: ART
  MORPH: Case=Acc|Definite=Def|Gender=Neut|Number=Sing|PronType=Art

Buch
  POS: NOUN
  TAG: NN
  MORPH: Case=Acc|Gender=Neut|Number=Sing

.
  POS: PUNCT
  TAG: $.
  MORPH: 



In [2]:
import itertools
from typing import Generator

from turkish_morphology import analysis_pb2
from turkish_morphology import analyze
from turkish_morphology import decompose

from absl import app


def _analyze(token: str) -> Generator[analysis_pb2.Analysis, None, None]:
  human_readables = analyze.surface_form(token, use_proper_feature=False)
  yield from map(decompose.human_readable_analysis, human_readables)


def main(unused_argv):
  sentence = "Ayşe eve geldiğinde Ali gitmişti"
  tokens = sentence.split()

  analyses = itertools.chain.from_iterable(_analyze(t) for t in tokens)
  print('-' * 20)
  print(analyses)
  print('-' * 20)
  word_stems = set(a.ig[0].root.morpheme.lower() for a in analyses)

  print("Unique word stems that appear in morphological analyses of the"
        " sentence '{}'\n".format(sentence))

  for word_stem in sorted(word_stems):
    print(word_stem)


if __name__ == "__main__":
  app.run(main)

ImportError: /lib64/libc.so.6: version `GLIBC_2.32' not found (required by /home/hpc/b279bb/b279bb15/miniconda3/envs/morphological/lib/python3.11/site-packages/external/org_openfst/pywrapfst.so)

In [5]:
from morphological_analyzer.TurkishMorphologicalAnalysis-Py.TurkishMorphologicalAnalysis import FsmMorphologicalAnalyzer
fsm = FsmMorphologicalAnalyzer()
word = "yarına"
fsmParseList = fsm.morphologicalAnalysis(word)
for i in range(fsmParseList.size()):
  	print(fsmParseList.getFsmParse(i).transitionList())

SyntaxError: invalid syntax (3404266310.py, line 1)